# **Import Library**

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import avg, col
from pyspark.sql.functions import month, year, sum as sum_
from pyspark.sql.functions import sum as sum_
from pyspark.sql.functions import col, sum as sum_, count
import matplotlib.pyplot as plt
from pyspark.sql.functions import sum
import pandas as pd
from pyspark.sql.functions import when
from pyspark.sql.functions import lit
import numpy as np
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.evaluation import ClusteringEvaluator
import seaborn as sns
import mlflow

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:632)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$5(UsageLogging.scala:659)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:147)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:349)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Att

# Data Loading and Import

In this step, the dataset is loaded into the Databricks environment using Apache Spark. The dataset is read from the specified file path in CSV format, with headers enabled and schema automatically inferred.

This process converts the raw dataset into a Spark DataFrame, which allows efficient distributed data processing and analysis.

In [0]:
df = spark.read.csv("/Workspace/Users/vpwwickramarathna@gmail.com/Dataset/online_retail_II.csv", header=True, inferSchema=True)

# Dataset Overview

After loading the dataset, an initial exploration is performed to understand its structure, size, and attributes. This step is important to gain a clear understanding of the data before proceeding with preprocessing and analysis.

In [0]:
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("Columns:", df.columns)

Row count: 1067371
Column count: 8
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [0]:
display(df.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


## Classification Evaluation

In [0]:

# Classification Evaluation
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

results_classification = [
    ["DecisionTree", evaluator_acc.evaluate(dt_preds), evaluator_prec.evaluate(dt_preds), evaluator_rec.evaluate(dt_preds)],
    ["RandomForest", evaluator_acc.evaluate(rf_preds), evaluator_prec.evaluate(rf_preds), evaluator_rec.evaluate(rf_preds)],
    ["LogisticRegression", evaluator_acc.evaluate(lr_preds), evaluator_prec.evaluate(lr_preds), evaluator_rec.evaluate(lr_preds)]
]

##  Regression Evaluation

In [0]:

# Regression Evaluation
reg_evaluator_mae = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="mae")
reg_evaluator_rmse = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="rmse")
results_regression = [
    ["LinearRegression", reg_evaluator_mae.evaluate(lr_reg_preds), reg_evaluator_rmse.evaluate(lr_reg_preds)]
]

## Clustering Evaluation

In [0]:
# Clustering Evaluation
clust_evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
wcss = kmeans_model.summary.trainingCost
silhouette = clust_evaluator.evaluate(kmeans_preds)
results_clustering = [
    ["KMeans", wcss, silhouette]
]

## Display evaluation tables

In [0]:
# Display evaluation tables
classification_df = pd.DataFrame(results_classification, columns=["Model", "Accuracy", "Precision", "Recall"])
regression_df = pd.DataFrame(results_regression, columns=["Model", "MAE", "RMSE"])
clustering_df = pd.DataFrame(results_clustering, columns=["Model", "WCSS", "SilhouetteScore"])

display(classification_df)
display(regression_df)
display(clustering_df)

Model,Accuracy,Precision,Recall
DecisionTree,0.9903856115704192,0.9904007071398473,0.9903856115704192
RandomForest,0.9891589482190589,0.9891673446649574,0.9891589482190589
LogisticRegression,0.9535608462319471,0.9491191134497607,0.9535608462319471


Model,MAE,RMSE
LinearRegression,13.506619500381658,217.01930050649318


Model,WCSS,SilhouetteScore
KMeans,1.066084019780523E10,0.9999950077298911


## Evaluate Random Forest model accuracy on test data

In [0]:
# Evaluate Random Forest model accuracy on test data
rf_test_accuracy = evaluator_acc.evaluate(rf_preds)
print("Random Forest Test Accuracy:", rf_test_accuracy)

classification_df = pd.DataFrame(results_classification, columns=["Model", "Accuracy", "Precision", "Recall"])
regression_df = pd.DataFrame(results_regression, columns=["Model", "MAE", "RMSE"])
clustering_df = pd.DataFrame(results_clustering, columns=["Model", "WCSS", "SilhouetteScore"])

display(classification_df)
display(regression_df)
display(clustering_df)

Random Forest Test Accuracy: 0.9891589482190589


Model,Accuracy,Precision,Recall
DecisionTree,0.9903856115704192,0.9904007071398473,0.9903856115704192
RandomForest,0.9891589482190589,0.9891673446649574,0.9891589482190589
LogisticRegression,0.9535608462319471,0.9491191134497607,0.9535608462319471


Model,MAE,RMSE
LinearRegression,13.506619500381658,217.01930050649318


Model,WCSS,SilhouetteScore
KMeans,1.066084019780523E10,0.9999950077298911


## Evaluate Decision Tree model accuracy on test data

In [0]:
# Evaluate Decision Tree model accuracy on test data
dt_test_accuracy = evaluator_acc.evaluate(dt_preds)
print("Best Model:", dt_test_accuracy)

Best Model: 0.9903856115704192
